# Lab 10: check Gemini's labels, then use them

Does the author of an AITA post admit fault? Run the short code cells, read the results, and discuss the questions with a partner.

The code is filled in. The main lab uses saved Gemini responses from November 12, 2026, so it does not need an API key or make new API calls. These are real model responses, not synthetic labels. Live calls are optional at the end.

The two label lists in part 2 are examples, not independently collected human responses.

In [1]:
from pathlib import Path
import pandas as pd

DATA = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/"
if Path("../data/week10_post_labels.csv").is_file():
    DATA = "../data/"
elif Path("data/week10_post_labels.csv").is_file():
    DATA = "data/"

## 1. Load the hand-labeled posts and set a baseline

`week10_post_labels.csv` has 30 posts, 15 NTA and 15 YTA. The `hand` column is a hand label for this definition:

- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

In [2]:
labels = pd.read_csv(DATA + "week10_post_labels.csv")
labels["hand"].value_counts()

hand
no     26
yes     4
Name: count, dtype: int64

In [3]:
baseline = (labels["hand"] == "no").mean()
baseline

np.float64(0.8666666666666667)

Why does always guessing no get such a high accuracy? Does it find any admissions of fault?

## 2. Compare two sets of labels

Read a post with your partner. Does the author admit fault? The example lists below contain two possible readings of the first eight posts. You can replace them with your own labels; label independently before comparing.

In [4]:
to_label = labels.head(8)
to_label[["title", "text"]]

,title,text
0,AITA for getting angry at my parents after the...,"I (25F) have two children, my son (4) and my d..."
1,AITA for only buying my daughter books that ar...,My daughter Rose (9) is a very advanced reader...
2,AITA because I told my son that I don't believ...,Son got himself a girlfriend about 6/7 months ...
3,AITA for getting mad at my parents after they ...,When I (16F) was 4 years old I was diagnosed w...
4,AITA for refusing to call my unborn nephew by ...,It’s aryan. \n \nNot arian. Not Erian. \n \nA...
5,AITA for not letting my teen daughter accept a...,"My son Adam has been dating a girl, who I'll c..."
6,AITA for switching doctors?,I (24F) currently see a small family doctor in...
7,AITA for not getting a traditional engagement ...,I (29M) just proposed to my girlfriend (27F) o...


Read one complete post below. Change 2 to another row number from 0 to 7 to read a different post.

In [5]:
print(to_label.iloc[2]["text"])

Son got himself a girlfriend about 6/7 months ago, I have no complaints about the girl, she's nice enough. He brought her home to meet me and my wife, everything was fine. My issue with it is that 2 weeks ago he announced that he would ask her to marry him. I thought to myself that he just had this plan for the future but he informed us that he wants to do it by next month.

My wife was over the moon with the news but I wasn't in the least. I told him that I don't believe this will work because they barely know each other, not only that but they're completely opposites. The girl is outgoing and charismatic while my son barely has enough friends, she makes him look like a dead weight not to mention that she's completely out of his league. Of course he got mad at this and so did my wife.

Now I found out that he won't be visiting us anymore and my wife has decided by herself that she will be visiting him instead. I don't see how it was wrong for me to give my opinion, it was just that, a

In [6]:
# Example labels, not independently collected responses.
# A reads the son's-marriage edit literally; B reads it as sarcastic.
person_a = ["no", "no", "yes", "no", "no", "yes", "no", "yes"]
person_b = ["no", "no", "no", "no", "no", "yes", "no", "yes"]
len(person_a), len(person_b)

(8, 8)

In [7]:
# Example: agreement, the share of posts the two of you gave the same label
ab = pd.DataFrame({"a": person_a, "b": person_b, "hand": to_label["hand"].values})
(ab["a"] == ab["b"]).mean()

np.float64(0.875)

In [8]:
# Agreement between A and the supplied labels
(ab["a"] == ab["hand"]).mean()

np.float64(0.875)

In [9]:
# Agreement between B and the supplied labels
(ab["b"] == ab["hand"]).mean()

np.float64(1.0)

Which post do the example labels disagree on? Is the author admitting fault, quoting someone else, or being sarcastic? Add one sentence to the definition to make it clearer.

## 3. Check Gemini on the 30 posts

Gemini received the same yes/no definition and was asked for a one-sentence reason. Load its saved responses and match them to the posts by ID. A response can have the right format and still be wrong.

In [10]:
gemini = pd.read_csv(DATA + "lab10_gemini_30.csv").merge(labels, on="id")
len(gemini)

30

In [11]:
accuracy = (gemini["llm_label"] == gemini["hand"]).mean()
accuracy

np.float64(0.8666666666666667)

In [12]:
# Difference from always guessing no
accuracy - baseline

np.float64(0.0)

In [13]:
confusion = pd.crosstab(gemini["hand"], gemini["llm_label"])
confusion

llm_label,no,yes
hand,,
no,23,3
yes,1,3


A false positive is a yes prediction with a no reference label. A false negative is the reverse.

In [14]:
false_positives = ((gemini["hand"] == "no") & (gemini["llm_label"] == "yes")).sum()
false_positives

np.int64(3)

In [15]:
false_negatives = ((gemini["hand"] == "yes") & (gemini["llm_label"] == "no")).sum()
false_negatives

np.int64(1)

Gemini and the always-no baseline have the same accuracy in this run. Do they make the same mistakes? Which one finds more of the four admissions?

In [16]:
# The file's gemini column is an earlier run with the same instruction. How many of your labels differ from it?
(gemini["llm_label"] != gemini["gemini"]).sum()  # Check: usually 0 to 3

np.int64(3)

In [17]:
mistakes = gemini[gemini["llm_label"] != gemini["hand"]]
mistakes[["title", "hand", "llm_label", "llm_reason"]]

,title,hand,llm_label,llm_reason
13,WIBTA for telling the bride I’m backing out as...,no,yes,The poster states in their edit that they alre...
21,AITA for not paying for the meal?,no,yes,The poster explicitly states that the only par...
23,AITA for cutting my best friend out my wedding?,no,yes,The poster admits that their decision to forgo...
25,AITA for inappropriately replying after two gi...,yes,no,The poster maintains throughout the text that ...


Does Gemini make more false positives or false negatives? Read the bridesmaid post below. Does changing a decision necessarily mean admitting fault?

In [18]:
case = gemini[gemini["id"] == "t3_ep1uwm"].iloc[0]
print(case["text"])

Long story short the bride was supposed to have her wedding locally so when she asked if we mind paying for our dress we said ok. Then she changed her mind 7 months before her wedding and decided to make it a destination wedding. When we asked about accommodations, flights, transportation if she will be offering to cover for it or even a portion. She said nope you’re on your own. Then we asked about the dress , hair and makeup apparently this is also all on us. So now one of the bridesmaids and I are calculating how much it would be just to be at her wedding it’s starting to rack up to  almost $1000. 

So would I be the asshole for backing out of her wedding since it’s just too expensive and we all have bills of our own.

Edit:
Wow this blew up.. I decided I’ve already said yes to be part of her wedding so I might as well commit to it. I rather spend money than lose years of friendship over money. Thank you everyone!


In [19]:
print(case["llm_reason"])

The poster states in their edit that they already made a commitment and chose to stick with it despite the costs.


## 4. Label a separate sample

`week10_posts.csv` has 785 posts with a verdict but no hand label. Select 20 NTA and 20 YTA posts, excluding the 30 used above, and compare the shares that Gemini labels yes.

In [20]:
posts = pd.read_csv(DATA + "week10_posts.csv")
unseen = posts[~posts["id"].isin(labels["id"])]
sample = unseen.groupby("verdict").sample(n=20, random_state=211)
sample["verdict"].value_counts()

verdict
NTA    20
YTA    20
Name: count, dtype: int64

In [21]:
new = pd.read_csv(DATA + "lab10_gemini_new.csv").merge(sample, on="id")
len(new)

40

In [22]:
shares = pd.crosstab(new["verdict"], new["llm_label"], normalize="index")
shares

llm_label,no,yes
verdict,,
NTA,0.8,0.2
YTA,0.7,0.3


In [23]:
# Difference in the predicted admission shares
gap = shares.loc["YTA", "yes"] - shares.loc["NTA", "yes"]
gap

np.float64(0.09999999999999998)

Which group has more predicted admissions? One post is five percentage points here. Could a few labeling errors change the comparison? What would you check before claiming that one group really admits fault more often?

## 5. Describe what you found

Write three sentences:

1. How does Gemini compare with always guessing no?
2. What kind of mistake did you find?
3. Would you trust these labels to compare NTA and YTA posts? Why?

## Optional: agreement, precision, and recall

Kappa adjusts agreement for the agreement expected from how often each person uses each label. Recall is the share of reference admissions found. Precision is the share of yes predictions that match the reference.

In [24]:
from sklearn.metrics import cohen_kappa_score

cohen_kappa_score(ab["a"], ab["b"])

np.float64(0.7142857142857143)

In [25]:
true_positives = ((gemini["hand"] == "yes") & (gemini["llm_label"] == "yes")).sum()
recall = true_positives / (gemini["hand"] == "yes").sum()
recall

np.float64(0.75)

In [26]:
precision = true_positives / (gemini["llm_label"] == "yes").sum()
precision

np.float64(0.5)

What do precision and recall tell you that accuracy alone does not?

## Optional: compare a revised prompt

The revised prompt requires the author to endorse that their own action was wrong. Questions, reported criticism, and sarcasm alone do not count as admissions. It was informed by the same evaluation examples.

Load the saved revised responses and compare them with the first run.

In [27]:
gemini_v2 = pd.read_csv(DATA + "lab10_gemini_30_v2.csv").merge(labels, on="id")
len(gemini_v2)

30

In [28]:
# Match the two runs by post ID
paired = gemini[["id", "hand", "llm_label"]].merge(
    gemini_v2[["id", "llm_label"]],
    on="id",
    suffixes=("_v1", "_v2")
)
changed = paired["llm_label_v1"] != paired["llm_label_v2"]
changed.sum()

np.int64(3)

In [29]:
accuracy_v2 = (paired["llm_label_v2"] == paired["hand"]).mean()
accuracy_v2

np.float64(0.9)

In [30]:
# Read the posts whose predictions changed
paired[changed]

,id,hand,llm_label_v1,llm_label_v2
7,t3_jbxcsk,yes,yes,no
13,t3_ep1uwm,no,yes,no
21,t3_bf892b,no,yes,no


Did the revised prompt fix every mistake? Did it introduce any? Why would you need a separate human-labeled test set to evaluate the revision fairly?

## Optional: make your own live calls

Everything above runs without this section. To try the API, put your Gemini key in your local `.env` file and set `RUN_LIVE = True` below. The first two batches request 70 labels; the revised prompt adds 30 more. Requests can take several minutes and may incur API charges.

Run the supplied setup and helper cells; you do not need to type them. Results are written to new local CSV files, leaving the saved class examples unchanged. Keep your key out of notebook cells and outputs.

In [31]:
RUN_LIVE = False

In [32]:
if RUN_LIVE:
    from pathlib import Path
    import json
    import logging
    import os
    import time
    
    import pandas as pd
    from sklearn.metrics import cohen_kappa_score
    from dotenv import load_dotenv
    from google import genai
    from google.genai import errors, types
    
    for folder in (Path.cwd(), *Path.cwd().parents):
        env_path = folder / ".env"
        if env_path.is_file():
            load_dotenv(env_path, override=True)
            break
    
    key = os.environ.get("GEMINI_API_KEY", "")
    if key.startswith("paste-"):
        raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")
    
    client = genai.Client(api_key=key) if key else None
    GEMINI_MODEL = "gemini-3.5-flash-lite"
    
    logging.getLogger("google_genai.types").setLevel(logging.ERROR)  # a warning google-genai prints on every call

In [33]:
SYSTEM = """You read posts from the Reddit forum r/AmItheAsshole, where people describe a conflict and ask whether they were in the wrong.

Label whether the poster admits that they did something wrong.
- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

Give a reason of one sentence."""

if RUN_LIVE:
    CONFIG = types.GenerateContentConfig(
        system_instruction=SYSTEM,
        response_mime_type="application/json",
        response_schema={
            "type": "object",
            "properties": {
                "llm_label": {"type": "string", "enum": ["yes", "no"]},
                "llm_reason": {"type": "string"},
            },
            "required": ["llm_label", "llm_reason"],
        },
    )

In [34]:
if RUN_LIVE:
    # Send one prompt, and try again after error 429 or 503.
    def ask(prompt, config, max_tries=4):
        if client is None:
            raise RuntimeError("Set GEMINI_API_KEY to generate missing responses.")
        wait = 10
        for attempt in range(1, max_tries + 1):
            try:
                response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt, config=config)
                time.sleep(6)  # stay under the free limit of calls per minute
                answer = json.loads(response.text)
                if answer.get("llm_label") not in {"yes", "no"} or not isinstance(answer.get("llm_reason"), str) or not answer["llm_reason"].strip():
                    raise ValueError("Invalid structured response")
                return answer
            except errors.APIError as error:
                if error.code not in {429, 503} or attempt == max_tries:
                    raise
                print(error.code, "retrying in", wait, "seconds")
                time.sleep(wait)
                wait = wait * 2

In [35]:
if RUN_LIVE:
    # Label every post in a table and return the labels next to the table.
    # Each answer is saved to the file right away. If a cell that uses this stops, run it again: posts already in the file are skipped.
    def label_all(table, path, config):
        import hashlib
        path = Path(path)
        fingerprint = hashlib.sha256((GEMINI_MODEL + config.model_dump_json()).encode()).hexdigest()
        if path.exists():
            cached = pd.read_csv(path)
            if "config_sha256" not in cached or not cached["config_sha256"].eq(fingerprint).all():
                raise ValueError("Cache belongs to a different configuration; choose a new filename.")
        done = set(pd.read_csv(path)["id"]) if path.exists() else set()
        for row in table.itertuples():
            if row.id in done:
                continue
            answer = ask(f"Title: {row.title}\n\n{row.text}", config)
            line = pd.DataFrame([{"id": row.id, **answer, "model": GEMINI_MODEL, "config_sha256": fingerprint}])
            line.to_csv(path, mode="a", header=not path.exists(), index=False)
            done.add(row.id)
            print(len(done), "of", len(table), "done")
        result = pd.read_csv(path).merge(table, on="id", validate="one_to_one")
        if len(result) != len(table):
            raise ValueError("Some requested IDs have no prediction")
        return result

In [36]:
if RUN_LIVE:
    # Clarification aimed at false admissions caused by questions, sarcasm, or others' criticism.
    SYSTEM2 = SYSTEM + "\nCount an admission only when the author endorses that their own action was wrong. Asking whether they were wrong, reporting someone else's criticism, or sarcastically repeating criticism is not by itself an admission. A genuine admission still counts even if the author also defends other actions."
    CONFIG2 = CONFIG.model_copy(update={"system_instruction": SYSTEM2})
    SYSTEM2

In [37]:
if RUN_LIVE:
    live = label_all(labels, "lab10_my_30.csv", CONFIG)
    live_new = label_all(sample, "lab10_my_new.csv", CONFIG)
    live_v2 = label_all(labels, "lab10_my_30_v2.csv", CONFIG2)

Compare your live labels with the saved responses. Which predictions changed? Use the same short calculations from the main lab to check your results.